# CNN-EXP-006 — BPE test

Меняем только выход модели: общий backbone из CNN-EXP-001 получает две головы.
Presence предсказывает `count > 0`, intensity будет предсказывать `log1p(count)` на положительных позициях.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import torch
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)

Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))

DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DIRECTML DILATED CONV1D STRESS TEST
# ============================================================

import gc
import torch
import torch.nn as nn
import torch_directml

device = torch_directml.device()

DILATIONS_TO_TEST = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)

B = 4
C = 32
L = 3708

print("Testing DirectML Conv1d dilations")
print("=" * 80)

for dilation in DILATIONS_TO_TEST:

    print(
        f"START dilation={dilation}",
        flush=True,
    )

    x = torch.randn(
        B,
        C,
        L,
        dtype=torch.float32,
    ).to(device)

    conv = nn.Conv1d(
        C,
        C,
        kernel_size=3,
        dilation=dilation,
        padding=dilation,
    ).to(device)

    with torch.no_grad():
        y = conv(x)

        # заставляем реально использовать результат
        value = y.mean().cpu().item()

    print(
        f"OK    dilation={dilation} "
        f"| shape={tuple(y.shape)} "
        f"| mean={value:.6f}",
        flush=True,
    )

    del x, y, conv
    gc.collect()

print("=" * 80)
print("ALL DILATIONS PASSED")

Testing DirectML Conv1d dilations
START dilation=1
OK    dilation=1 | shape=(4, 32, 3708) | mean=-0.008110
START dilation=2
OK    dilation=2 | shape=(4, 32, 3708) | mean=-0.002918
START dilation=4
OK    dilation=4 | shape=(4, 32, 3708) | mean=-0.001077
START dilation=8
OK    dilation=8 | shape=(4, 32, 3708) | mean=0.000125
START dilation=16
OK    dilation=16 | shape=(4, 32, 3708) | mean=-0.000934
START dilation=32
OK    dilation=32 | shape=(4, 32, 3708) | mean=0.004549
START dilation=64
OK    dilation=64 | shape=(4, 32, 3708) | mean=0.027822
START dilation=128
OK    dilation=128 | shape=(4, 32, 3708) | mean=0.002551
START dilation=256
OK    dilation=256 | shape=(4, 32, 3708) | mean=0.005076
START dilation=512
OK    dilation=512 | shape=(4, 32, 3708) | mean=-0.000635
ALL DILATIONS PASSED


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

assert train_contigs.isdisjoint(validation_contigs)

display(split.summary)
print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3


In [4]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(batch_size=4)

print("Обычный вход:", train_batch.x.shape)
print("Обе ориентации:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Положительных:", int(train_batch.target.sum()))
print("Sampling branches:", train_batch.sampling_branch)

Обычный вход: torch.Size([4, 5, 1284])
Обе ориентации: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Положительных: 33
Sampling branches: ('negative', 'positive', 'positive', 'negative')


In [6]:
print("=" * 80)
print("PROFILE TILES INSPECTION")
print("=" * 80)

print("Тип tiles:")
print(type(train_batch.tiles))

print("\nShape:")
print(train_batch.tiles.shape)

print("\nColumns:")
print(train_batch.tiles.columns.tolist())

print("\nПервая строка:")
tile = train_batch.tiles.iloc[0]

display(tile)

print("\nЗначения первой строки:")
for key, value in tile.items():
    print(f"{key:<25} = {value}")

print("\nРазмер обычного CNN-входа:")
print("x shape:", train_batch.x[0].shape)

print("\nProfile config:")
print(profile_config)

PROFILE TILES INSPECTION
Тип tiles:
<class 'pandas.DataFrame'>

Shape:
(4, 13)

Columns:
['tile_id', 'contig', 'target_start', 'allowed_bp', 'positive_positions', 'count_sum', 'target_end', 'allowed_position_strand', 'negative_positions', 'q_positive', 'q_negative', 'q_mixture', 'sampling_branch']

Первая строка:


tile_id                         111696
contig                     NC_064047.1
target_start                   5334016
allowed_bp                         498
positive_positions                   0
count_sum                            0
target_end                     5335040
allowed_position_strand            996
negative_positions                 996
q_positive                         0.0
q_negative                    0.000004
q_mixture                     0.000002
sampling_branch               negative
Name: 0, dtype: object


Значения первой строки:
tile_id                   = 111696
contig                    = NC_064047.1
target_start              = 5334016
allowed_bp                = 498
positive_positions        = 0
count_sum                 = 0
target_end                = 5335040
allowed_position_strand   = 996
negative_positions        = 996
q_positive                = 0.0
q_negative                = 4.370458357129696e-06
q_mixture                 = 2.185229178564848e-06
sampling_branch           = negative

Размер обычного CNN-входа:
x shape: torch.Size([5, 1284])

Profile config:
ProfileWindowConfig(target_length=1024, context_flank=130, positive_branch_fraction=0.5)


In [8]:
# ============================================================
# 5B. Восстанавливаем ТОЧНОЕ genomic window — TORCH ONLY
# ============================================================

tile = train_batch.tiles.iloc[0]

contig = str(tile["contig"])
target_start = int(tile["target_start"])
target_end = int(tile["target_end"])

flank = profile_config.context_flank

input_start = target_start - flank
input_end = target_end + flank


# ------------------------------------------------------------
# Из genome storage сразу делаем torch.Tensor
# ------------------------------------------------------------

raw_window = torch.as_tensor(
    sequences[contig][input_start:input_end],
    dtype=torch.long,
).clone()


# ------------------------------------------------------------
# Lookup: A/a -> A, C/c -> C, G/g -> G, T/t -> T, N -> N
# ------------------------------------------------------------

BASE_LOOKUP = torch.tensor(
    [0, 1, 2, 3, 0, 1, 2, 3, 4],
    dtype=torch.long,
)

collapsed_window = BASE_LOOKUP[raw_window]


# ------------------------------------------------------------
# Softmask
# ------------------------------------------------------------

SOFTMASK_LOOKUP = torch.tensor(
    [0, 0, 0, 0, 1, 1, 1, 1, 0],
    dtype=torch.long,
)

softmask_window = SOFTMASK_LOOKUP[raw_window]


# ------------------------------------------------------------
# Что реально лежит в CNN batch
# [5, 1284] -> [1284]
# ------------------------------------------------------------

cnn_codes = (
    train_batch.x[0]
    .argmax(dim=0)
    .to(dtype=torch.long)
    .cpu()
)


print("=" * 80)
print("EXACT WINDOW CHECK — TORCH")
print("=" * 80)

print(f"Contig:              {contig}")
print(f"Target:              [{target_start:,}, {target_end:,})")
print(f"Input:               [{input_start:,}, {input_end:,})")

print()
print(f"raw_window:          {raw_window.shape} {raw_window.dtype}")
print(f"collapsed_window:    {collapsed_window.shape} {collapsed_window.dtype}")
print(f"softmask_window:     {softmask_window.shape} {softmask_window.dtype}")
print(f"cnn_codes:           {cnn_codes.shape} {cnn_codes.dtype}")

print()
print(
    "DNA matches CNN:    ",
    torch.equal(collapsed_window, cnn_codes)
)

print(
    "Softmask bp:        ",
    int(softmask_window.sum().item())
)

print(
    "Softmask fraction:  ",
    f"{softmask_window.float().mean().item():.4%}"
)

print("\nПервые 50 collapsed codes:")
print(collapsed_window[:50])

print("\nПервые 50 CNN codes:")
print(cnn_codes[:50])

EXACT WINDOW CHECK — TORCH
Contig:              NC_064047.1
Target:              [5,334,016, 5,335,040)
Input:               [5,333,886, 5,335,170)

raw_window:          torch.Size([1284]) torch.int64
collapsed_window:    torch.Size([1284]) torch.int64
softmask_window:     torch.Size([1284]) torch.int64
cnn_codes:           torch.Size([1284]) torch.int64

DNA matches CNN:     True
Softmask bp:         226
Softmask fraction:   17.6012%

Первые 50 collapsed codes:
tensor([2, 2, 2, 3, 2, 3, 3, 2, 3, 3, 2, 1, 1, 0, 2, 2, 0, 2, 3, 2, 0, 1, 0, 3,
        3, 3, 0, 1, 1, 3, 2, 0, 1, 0, 2, 2, 2, 0, 0, 2, 2, 3, 0, 1, 0, 0, 2, 3,
        2, 3])

Первые 50 CNN codes:
tensor([2, 2, 2, 3, 2, 3, 3, 2, 3, 3, 2, 1, 1, 0, 2, 2, 0, 2, 3, 2, 0, 1, 0, 3,
        3, 3, 0, 1, 1, 3, 2, 0, 1, 0, 2, 2, 2, 0, 0, 2, 2, 3, 0, 1, 0, 0, 2, 3,
        2, 3])


In [10]:
# ============================================================
# 5C. BPE-50 TOKENIZATION ДЛЯ РЕАЛЬНОГО CNN WINDOW
# TORCH ONLY
# ============================================================

BPE_PATH = PROJECT_ROOT / "artifacts" / "bpe_softmask_50.pt"

bpe_artifact = torch.load(
    BPE_PATH,
    map_location="cpu",
    weights_only=False,
)

merge_history = bpe_artifact["merge_history"]
token_text = bpe_artifact["token_text"]
token_length = bpe_artifact["token_length"]
vocab_size = bpe_artifact["vocab_size"]


def apply_bpe_torch(
    sequence: torch.Tensor,
    lowercase_count: torch.Tensor,
    merge_rules,
):
    """
    sequence:
        [L] token IDs

    lowercase_count:
        [L] число softmasked bp внутри каждого текущего токена

    Возвращает:
        bpe_tokens
        bpe_lowercase_count
    """

    tokens = sequence.clone()
    lower = lowercase_count.clone()

    for left, right, new_token in merge_rules:

        left = int(left)
        right = int(right)
        new_token = int(new_token)

        matches = (
            (tokens[:-1] == left)
            &
            (tokens[1:] == right)
        )

        starts = torch.nonzero(
            matches,
            as_tuple=False,
        ).flatten()

        if starts.numel() == 0:
            continue

        # ----------------------------------------------------
        # Для X + X убираем overlapping merges
        #
        # например:
        # T T T T
        #
        # starts = [0, 1, 2]
        # берём   = [0, 2]
        # ----------------------------------------------------

        if left == right and starts.numel() > 1:

            new_run = torch.cat([
                torch.tensor(
                    [True],
                    dtype=torch.bool,
                    device=starts.device,
                ),
                starts[1:] != starts[:-1] + 1,
            ])

            run_ids = torch.cumsum(
                new_run.to(torch.long),
                dim=0,
            ) - 1

            run_start_indices = torch.nonzero(
                new_run,
                as_tuple=False,
            ).flatten()

            start_index_for_each_run = (
                run_start_indices[run_ids]
            )

            offsets = (
                torch.arange(
                    starts.numel(),
                    device=starts.device,
                )
                -
                start_index_for_each_run
            )

            starts = starts[
                offsets % 2 == 0
            ]

        # ----------------------------------------------------
        # Merge token IDs
        # ----------------------------------------------------

        new_tokens = tokens.clone()

        new_tokens[starts] = new_token

        # ----------------------------------------------------
        # Merge softmask counts
        # ----------------------------------------------------

        new_lower = lower.clone()

        new_lower[starts] = (
            lower[starts]
            +
            lower[starts + 1]
        )

        # второй token каждой пары удаляем
        remove = torch.zeros(
            tokens.shape[0],
            dtype=torch.bool,
            device=tokens.device,
        )

        remove[starts + 1] = True

        tokens = new_tokens[~remove]
        lower = new_lower[~remove]

    return tokens, lower


# ============================================================
# Применяем BPE-50
# ============================================================

bpe_tokens, bpe_lower_count = apply_bpe_torch(
    collapsed_window,
    softmask_window,
    merge_history,
)


# ------------------------------------------------------------
# Token length lookup
# ------------------------------------------------------------

token_length_lookup = torch.zeros(
    vocab_size,
    dtype=torch.long,
)

for token_id, length in token_length.items():
    token_length_lookup[int(token_id)] = int(length)


bpe_lengths = token_length_lookup[bpe_tokens]

bpe_softmask_fraction = (
    bpe_lower_count.float()
    /
    bpe_lengths.float()
)


# ------------------------------------------------------------
# Точные bp-координаты каждого BPE token внутри окна
# ------------------------------------------------------------

bpe_end = torch.cumsum(
    bpe_lengths,
    dim=0,
)

bpe_start = (
    bpe_end
    -
    bpe_lengths
)


# ============================================================
# CHECKS
# ============================================================

print("=" * 80)
print("BPE-50 WINDOW TOKENIZATION")
print("=" * 80)

print(f"Original bp:             {collapsed_window.numel():,}")
print(f"BPE tokens:              {bpe_tokens.numel():,}")

print(
    f"Compression:             "
    f"{collapsed_window.numel() / bpe_tokens.numel():.4f}x"
)

print()
print("bpe_tokens shape:        ", bpe_tokens.shape)
print("bpe_lengths shape:       ", bpe_lengths.shape)
print("softmask shape:          ", bpe_softmask_fraction.shape)
print("bp_start shape:          ", bpe_start.shape)
print("bp_end shape:            ", bpe_end.shape)

print()
print(
    "All bp preserved:        ",
    int(bpe_lengths.sum().item())
    ==
    collapsed_window.numel()
)

print(
    "Softmask preserved:      ",
    int(bpe_lower_count.sum().item())
    ==
    int(softmask_window.sum().item())
)


print("\nFIRST 20 BPE TOKENS")
print("-" * 80)

for i in range(min(20, bpe_tokens.numel())):

    token_id = int(bpe_tokens[i].item())

    print(
        f"{i:>3} | "
        f"id={token_id:>2} | "
        f"{token_text[token_id]:<6} | "
        f"bp=[{bpe_start[i].item():>4}, "
        f"{bpe_end[i].item():>4}) | "
        f"len={bpe_lengths[i].item()} | "
        f"soft={bpe_softmask_fraction[i].item():.2f}"
    )

BPE-50 WINDOW TOKENIZATION
Original bp:             1,284
BPE tokens:              510
Compression:             2.5176x

bpe_tokens shape:         torch.Size([510])
bpe_lengths shape:        torch.Size([510])
softmask shape:           torch.Size([510])
bp_start shape:           torch.Size([510])
bp_end shape:             torch.Size([510])

All bp preserved:         True
Softmask preserved:       True

FIRST 20 BPE TOKENS
--------------------------------------------------------------------------------
  0 | id=39 | GGG    | bp=[   0,    3) | len=3 | soft=0.00
  1 | id= 7 | TG     | bp=[   3,    5) | len=2 | soft=0.00
  2 | id=25 | TTG    | bp=[   5,    8) | len=3 | soft=0.00
  3 | id=25 | TTG    | bp=[   8,   11) | len=3 | soft=0.00
  4 | id= 8 | CC     | bp=[  11,   13) | len=2 | soft=0.00
  5 | id=32 | AGG    | bp=[  13,   16) | len=3 | soft=0.00
  6 | id= 9 | AG     | bp=[  16,   18) | len=2 | soft=0.00
  7 | id= 7 | TG     | bp=[  18,   20) | len=2 | soft=0.00
  8 | id=10 | AC     |

In [11]:
# ============================================================
# 5D. BPE-50 ENCODING ДЛЯ ВСЕГО TRAIN BATCH
# TORCH ONLY
# ============================================================

PAD_TOKEN_ID = vocab_size


def encode_tile_bpe(tile):
    contig = str(tile["contig"])

    target_start = int(tile["target_start"])
    target_end = int(tile["target_end"])

    input_start = target_start - profile_config.context_flank
    input_end = target_end + profile_config.context_flank

    # --------------------------------------------------------
    # Genome -> torch
    # --------------------------------------------------------

    raw = torch.as_tensor(
        sequences[contig][input_start:input_end],
        dtype=torch.long,
    ).clone()

    collapsed = BASE_LOOKUP[raw]
    softmask = SOFTMASK_LOOKUP[raw]

    # --------------------------------------------------------
    # BPE
    # --------------------------------------------------------

    tokens, lower_count = apply_bpe_torch(
        collapsed,
        softmask,
        merge_history,
    )

    lengths = token_length_lookup[tokens]

    softmask_fraction = (
        lower_count.float()
        /
        lengths.float()
    )

    # --------------------------------------------------------
    # bp -> token mapping
    #
    # Например lengths = [3,2,4]
    #
    # bp_to_token =
    # [0,0,0, 1,1, 2,2,2,2]
    # --------------------------------------------------------

    bp_to_token = torch.repeat_interleave(
        torch.arange(
            tokens.numel(),
            dtype=torch.long,
        ),
        lengths,
    )

    assert bp_to_token.numel() == collapsed.numel()

    return {
        "tokens": tokens,
        "lengths": lengths,
        "softmask_fraction": softmask_fraction,
        "bp_to_token": bp_to_token,
    }


# ============================================================
# Кодируем весь batch
# ============================================================

encoded_batch = [
    encode_tile_bpe(train_batch.tiles.iloc[i])
    for i in range(len(train_batch.tiles))
]

batch_size = len(encoded_batch)

max_tokens = max(
    item["tokens"].numel()
    for item in encoded_batch
)

bp_length = train_batch.x.shape[-1]


# ============================================================
# Padded tensors
# ============================================================

bpe_token_ids = torch.full(
    (batch_size, max_tokens),
    PAD_TOKEN_ID,
    dtype=torch.long,
)

bpe_token_lengths = torch.zeros(
    (batch_size, max_tokens),
    dtype=torch.long,
)

bpe_softmask = torch.zeros(
    (batch_size, max_tokens),
    dtype=torch.float32,
)

bpe_token_mask = torch.zeros(
    (batch_size, max_tokens),
    dtype=torch.bool,
)

bpe_bp_to_token = torch.zeros(
    (batch_size, bp_length),
    dtype=torch.long,
)


for i, item in enumerate(encoded_batch):

    n = item["tokens"].numel()

    bpe_token_ids[i, :n] = item["tokens"]

    bpe_token_lengths[i, :n] = item["lengths"]

    bpe_softmask[i, :n] = item[
        "softmask_fraction"
    ]

    bpe_token_mask[i, :n] = True

    bpe_bp_to_token[i] = item[
        "bp_to_token"
    ]


# ============================================================
# DIAGNOSTICS
# ============================================================

print("=" * 80)
print("BPE BATCH")
print("=" * 80)

print("Original CNN batch:")
print("  ", train_batch.x.shape)

print("\nBPE tensors:")
print("token_ids:       ", bpe_token_ids.shape)
print("token_lengths:   ", bpe_token_lengths.shape)
print("softmask:        ", bpe_softmask.shape)
print("token_mask:      ", bpe_token_mask.shape)
print("bp_to_token:     ", bpe_bp_to_token.shape)


print("\nPER WINDOW")
print("-" * 80)

for i, item in enumerate(encoded_batch):

    n_tokens = item["tokens"].numel()

    compression = (
        bp_length
        /
        n_tokens
    )

    print(
        f"{i}: "
        f"{bp_length} bp -> "
        f"{n_tokens:4d} tokens "
        f"| {compression:.4f}x"
    )


print("\nPAD TOKEN ID:", PAD_TOKEN_ID)

print(
    "All bp mappings valid:",
    all(
        item["bp_to_token"].numel() == bp_length
        for item in encoded_batch
    )
)

BPE BATCH
Original CNN batch:
   torch.Size([4, 5, 1284])

BPE tensors:
token_ids:        torch.Size([4, 511])
token_lengths:    torch.Size([4, 511])
softmask:         torch.Size([4, 511])
token_mask:       torch.Size([4, 511])
bp_to_token:      torch.Size([4, 1284])

PER WINDOW
--------------------------------------------------------------------------------
0: 1284 bp ->  510 tokens | 2.5176x
1: 1284 bp ->  507 tokens | 2.5325x
2: 1284 bp ->  503 tokens | 2.5527x
3: 1284 bp ->  511 tokens | 2.5127x

PAD TOKEN ID: 55
All bp mappings valid: True


In [12]:
# ============================================================
# 5E. BPE ENCODER:
# token IDs -> embedding -> CNN -> обратно в base resolution
# ============================================================

import torch.nn as nn


class BPEContextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        embedding_dim: int = 16,
        hidden_channels: int = 32,
        max_token_length: int = 4,
    ):
        super().__init__()

        self.max_token_length = max_token_length

        # +1 потому что PAD имеет id == vocab_size
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size + 1,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id,
        )

        # embedding
        # + softmask_fraction
        # + normalized token length
        input_channels = embedding_dim + 2

        self.cnn = nn.Sequential(
            nn.Conv1d(
                input_channels,
                hidden_channels,
                kernel_size=5,
                padding=2,
            ),
            nn.GELU(),

            nn.Conv1d(
                hidden_channels,
                hidden_channels,
                kernel_size=5,
                padding=2,
            ),
            nn.GELU(),
        )

    def forward(
        self,
        token_ids,
        token_lengths,
        softmask_fraction,
        token_mask,
        bp_to_token,
    ):

        # ----------------------------------------------------
        # [B, T] -> [B, T, E]
        # ----------------------------------------------------

        embedding = self.embedding(token_ids)

        # ----------------------------------------------------
        # Дополнительные признаки
        # ----------------------------------------------------

        length_feature = (
            token_lengths.float()
            /
            self.max_token_length
        ).unsqueeze(-1)

        softmask_feature = (
            softmask_fraction
            .unsqueeze(-1)
        )

        # [B, T, E+2]
        x = torch.cat(
            [
                embedding,
                softmask_feature,
                length_feature,
            ],
            dim=-1,
        )

        # PAD позиции обнуляем
        x = (
            x
            *
            token_mask.unsqueeze(-1)
        )

        # Conv1d ожидает:
        # [B, channels, T]
        x = x.transpose(1, 2)

        # ----------------------------------------------------
        # CNN работает НА СЖАТОЙ ОСИ
        # ----------------------------------------------------

        token_features = self.cnn(x)

        # снова зануляем padded хвост
        token_features = (
            token_features
            *
            token_mask.unsqueeze(1)
        )

        # ----------------------------------------------------
        # BPE tokens -> base resolution
        #
        # token_features:
        # [B, C, T]
        #
        # bp_to_token:
        # [B, 1284]
        # ----------------------------------------------------

        gather_index = (
            bp_to_token
            .unsqueeze(1)
            .expand(
                -1,
                token_features.shape[1],
                -1,
            )
        )

        base_features = torch.gather(
            token_features,
            dim=2,
            index=gather_index,
        )

        return token_features, base_features


# ============================================================
# Создаём encoder
# ============================================================

bpe_encoder = BPEContextEncoder(
    vocab_size=vocab_size,
    pad_token_id=PAD_TOKEN_ID,
    embedding_dim=16,
    hidden_channels=32,
    max_token_length=max(token_length.values()),
)


# ============================================================
# FORWARD
# ============================================================

with torch.no_grad():

    token_features, base_features = bpe_encoder(
        bpe_token_ids,
        bpe_token_lengths,
        bpe_softmask,
        bpe_token_mask,
        bpe_bp_to_token,
    )


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("BPE CONTEXT ENCODER")
print("=" * 80)

print("Input DNA:")
print("  ", train_batch.x.shape)

print("\nCompressed BPE:")
print("  token_ids:      ", bpe_token_ids.shape)

print("\nCNN on BPE axis:")
print("  token_features: ", token_features.shape)

print("\nExpanded back to bp:")
print("  base_features:  ", base_features.shape)

print()
print(
    "Base length restored:",
    base_features.shape[-1] == train_batch.x.shape[-1]
)

print(
    "Channels produced:",
    base_features.shape[1]
)

BPE CONTEXT ENCODER
Input DNA:
   torch.Size([4, 5, 1284])

Compressed BPE:
  token_ids:       torch.Size([4, 511])

CNN on BPE axis:
  token_features:  torch.Size([4, 32, 511])

Expanded back to bp:
  base_features:   torch.Size([4, 32, 1284])

Base length restored: True
Channels produced: 32


In [13]:
# ============================================================
# 5F. INSPECT CURRENT MODEL ARCHITECTURE
# ============================================================

import inspect

from ml_project.epic_cnn import (
    CNNBackbone,
    CNNPresenceIntensity,
    create_cnn_presence_intensity,
)

print("=" * 80)
print("CNNBackbone")
print("=" * 80)

print(inspect.getsource(CNNBackbone))

print("\n" + "=" * 80)
print("CNNPresenceIntensity")
print("=" * 80)

print(inspect.getsource(CNNPresenceIntensity))

print("\n" + "=" * 80)
print("create_cnn_presence_intensity")
print("=" * 80)

print(inspect.getsource(create_cnn_presence_intensity))

CNNBackbone
class CNNBackbone(nn.Module):
    """Stem plus named residual stages with optional activation capture."""

    def __init__(self, stem: nn.Module, blocks: Sequence[nn.Module]) -> None:
        super().__init__()
        self.stem = stem
        self.blocks = nn.ModuleList(blocks)

    def forward(
        self,
        x: torch.Tensor,
        return_activations: bool = False,
    ):
        activations: dict[str, torch.Tensor] = {}
        x = self.stem(x)

        if return_activations:
            activations["stem"] = x

        for index, block in enumerate(self.blocks, start=1):
            x = block(x)
            if return_activations:
                activations[f"block{index}"] = x

        if return_activations:
            return x, activations
        return x


CNNPresenceIntensity
class CNNPresenceIntensity(nn.Module):
    """Shared backbone with presence and positive-count intensity outputs."""

    def __init__(self, backbone: CNNBackbone, profile_head: nn.

In [14]:
# ============================================================
# 5G. LOCAL + BPE FUSION MODEL
# Smoke test на прямой ориентации
# ============================================================

class CNNPresenceIntensityBPE(nn.Module):

    def __init__(
        self,
        local_model,
        bpe_encoder,
        local_channels=32,
        bpe_channels=32,
    ):
        super().__init__()

        # Существующий CNN backbone
        self.backbone = local_model.backbone

        # Наша новая BPE-ветка
        self.bpe_encoder = bpe_encoder

        # 32 local + 32 BPE = 64
        # затем учимся смешивать их обратно в 32
        self.fusion = nn.Sequential(
            nn.Conv1d(
                local_channels + bpe_channels,
                local_channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        # Та же логика двух выходов
        self.profile_head = nn.Conv1d(
            local_channels,
            2,
            kernel_size=1,
        )

    def forward(
        self,
        x,
        token_ids,
        token_lengths,
        softmask_fraction,
        token_mask,
        bp_to_token,
    ):

        # ----------------------------------------------------
        # LOCAL
        # ----------------------------------------------------

        local_features = self.backbone(x)

        # [B, 32, 1284]


        # ----------------------------------------------------
        # BPE GLOBAL
        # ----------------------------------------------------

        _, bpe_features = self.bpe_encoder(
            token_ids,
            token_lengths,
            softmask_fraction,
            token_mask,
            bp_to_token,
        )

        # [B, 32, 1284]


        # ----------------------------------------------------
        # FUSION
        # ----------------------------------------------------

        combined = torch.cat(
            [local_features, bpe_features],
            dim=1,
        )

        # [B, 64, 1284]

        fused = self.fusion(combined)

        # [B, 32, 1284]


        # ----------------------------------------------------
        # HEAD
        # ----------------------------------------------------

        outputs = self.profile_head(fused)

        presence = outputs[:, 0:1, :]
        intensity = outputs[:, 1:2, :]

        return presence, intensity


# ============================================================
# Создаём модель
# ============================================================

local_model = create_cnn_presence_intensity()

fusion_model = CNNPresenceIntensityBPE(
    local_model=local_model,
    bpe_encoder=bpe_encoder,
)


# ============================================================
# Smoke test
# ============================================================

with torch.no_grad():

    presence, intensity = fusion_model(
        train_batch.x,
        bpe_token_ids,
        bpe_token_lengths,
        bpe_softmask,
        bpe_token_mask,
        bpe_bp_to_token,
    )


print("=" * 80)
print("LOCAL + BPE FUSION")
print("=" * 80)

print("Input:")
print(" ", train_batch.x.shape)

print("\nOutputs:")
print(" presence: ", presence.shape)
print(" intensity:", intensity.shape)

print()
print(
    "Base resolution preserved:",
    presence.shape[-1] == train_batch.x.shape[-1]
)

LOCAL + BPE FUSION
Input:
  torch.Size([4, 5, 1284])

Outputs:
 presence:  torch.Size([4, 1, 1284])
 intensity: torch.Size([4, 1, 1284])

Base resolution preserved: True


In [15]:
# ============================================================
# 5H. PLUS + REVERSE-COMPLEMENT BPE BATCH
# ============================================================

RC_LOOKUP = torch.tensor(
    [
        3,  # A -> T
        2,  # C -> G
        1,  # G -> C
        0,  # T -> A
        7,  # a -> t
        6,  # c -> g
        5,  # g -> c
        4,  # t -> a
        8,  # N -> N
    ],
    dtype=torch.long,
)


def encode_raw_bpe(raw: torch.Tensor):

    collapsed = BASE_LOOKUP[raw]
    softmask = SOFTMASK_LOOKUP[raw]

    tokens, lower_count = apply_bpe_torch(
        collapsed,
        softmask,
        merge_history,
    )

    lengths = token_length_lookup[tokens]

    softmask_fraction = (
        lower_count.float()
        /
        lengths.float()
    )

    bp_to_token = torch.repeat_interleave(
        torch.arange(
            tokens.numel(),
            dtype=torch.long,
        ),
        lengths,
    )

    assert bp_to_token.numel() == raw.numel()

    return {
        "tokens": tokens,
        "lengths": lengths,
        "softmask_fraction": softmask_fraction,
        "bp_to_token": bp_to_token,
    }


def build_bpe_orientation_batch(reverse_complement=False):

    encoded = []

    for i in range(len(train_batch.tiles)):

        tile = train_batch.tiles.iloc[i]

        contig = str(tile["contig"])

        input_start = (
            int(tile["target_start"])
            -
            profile_config.context_flank
        )

        input_end = (
            int(tile["target_end"])
            +
            profile_config.context_flank
        )

        raw = torch.as_tensor(
            sequences[contig][input_start:input_end],
            dtype=torch.long,
        ).clone()

        if reverse_complement:
            raw = RC_LOOKUP[raw].flip(0)

        encoded.append(
            encode_raw_bpe(raw)
        )

    max_tokens = max(
        item["tokens"].numel()
        for item in encoded
    )

    batch_size = len(encoded)
    bp_length = profile_config.input_length

    token_ids = torch.full(
        (batch_size, max_tokens),
        PAD_TOKEN_ID,
        dtype=torch.long,
    )

    token_lengths = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.long,
    )

    softmask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.float32,
    )

    token_mask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.bool,
    )

    bp_to_token = torch.zeros(
        (batch_size, bp_length),
        dtype=torch.long,
    )

    for i, item in enumerate(encoded):

        n = item["tokens"].numel()

        token_ids[i, :n] = item["tokens"]
        token_lengths[i, :n] = item["lengths"]
        softmask[i, :n] = item["softmask_fraction"]
        token_mask[i, :n] = True
        bp_to_token[i] = item["bp_to_token"]

    return (
        token_ids,
        token_lengths,
        softmask,
        token_mask,
        bp_to_token,
    )


# ============================================================
# PLUS
# ============================================================

bpe_plus = build_bpe_orientation_batch(
    reverse_complement=False
)

# ============================================================
# REVERSE COMPLEMENT
# ============================================================

bpe_rc = build_bpe_orientation_batch(
    reverse_complement=True
)


print("=" * 80)
print("BPE STRAND BATCHES")
print("=" * 80)

print("PLUS")
print(" token_ids:", bpe_plus[0].shape)
print(" valid tokens:",
      bpe_plus[3].sum(dim=1))

print("\nREVERSE COMPLEMENT")
print(" token_ids:", bpe_rc[0].shape)
print(" valid tokens:",
      bpe_rc[3].sum(dim=1))

print()
print(
    "Both restore 1284 bp:",
    bpe_plus[4].shape[-1] == 1284
    and
    bpe_rc[4].shape[-1] == 1284
)

BPE STRAND BATCHES
PLUS
 token_ids: torch.Size([4, 511])
 valid tokens: tensor([510, 507, 503, 511])

REVERSE COMPLEMENT
 token_ids: torch.Size([4, 513])
 valid tokens: tensor([503, 510, 511, 513])

Both restore 1284 bp: True


In [16]:
# ============================================================
# 5I. VERIFY x_both_strands ORDER
# ============================================================

B = train_batch.x.shape[0]

print("=" * 80)
print("VERIFY STRAND ORDER")
print("=" * 80)

print("x shape:")
print(" ", train_batch.x.shape)

print("x_both_strands shape:")
print(" ", train_batch.x_both_strands.shape)


# ------------------------------------------------------------
# Строим reverse-complement прямо из one-hot CNN input
#
# channels:
# 0=A, 1=C, 2=G, 3=T, 4=N
#
# RC:
# A <-> T
# C <-> G
# N -> N
# ------------------------------------------------------------

rc_channel_order = torch.tensor(
    [3, 2, 1, 0, 4],
    dtype=torch.long,
)

expected_rc = (
    train_batch.x[:, rc_channel_order, :]
    .flip(-1)
)


# ------------------------------------------------------------
# Проверяем гипотезу:
#
# first B  = PLUS
# second B = RC
# ------------------------------------------------------------

first_half = train_batch.x_both_strands[:B]
second_half = train_batch.x_both_strands[B:]


print()
print(
    "First half == PLUS:",
    torch.equal(
        first_half.cpu(),
        train_batch.x.cpu(),
    )
)

print(
    "Second half == RC:",
    torch.equal(
        second_half.cpu(),
        expected_rc.cpu(),
    )
)

VERIFY STRAND ORDER
x shape:
  torch.Size([4, 5, 1284])
x_both_strands shape:
  torch.Size([8, 5, 1284])

First half == PLUS: True
Second half == RC: True


In [17]:
# ============================================================
# 5J. FULL STRAND-AWARE FORWARD
# LOCAL + BPE
# ============================================================

B = train_batch.x.shape[0]
flank = profile_config.context_flank

x_plus = train_batch.x_both_strands[:B]
x_rc = train_batch.x_both_strands[B:]


with torch.no_grad():

    # ========================================================
    # PLUS STRAND
    # ========================================================

    plus_presence_full, plus_intensity_full = fusion_model(
        x_plus,
        *bpe_plus,
    )


    # ========================================================
    # REVERSE-COMPLEMENT STRAND
    # ========================================================

    rc_presence_full, rc_intensity_full = fusion_model(
        x_rc,
        *bpe_rc,
    )


    # --------------------------------------------------------
    # RC output сейчас идёт в координатах reverse-complement.
    # Возвращаем его в координаты исходного genomic window.
    # --------------------------------------------------------

    minus_presence_full = rc_presence_full.flip(-1)
    minus_intensity_full = rc_intensity_full.flip(-1)


    # ========================================================
    # CENTRAL TARGET CROP
    #
    # 1284 bp:
    #
    # 130 context + 1024 target + 130 context
    # ========================================================

    plus_presence = plus_presence_full[
        :, :, flank:-flank
    ]

    plus_intensity = plus_intensity_full[
        :, :, flank:-flank
    ]

    minus_presence = minus_presence_full[
        :, :, flank:-flank
    ]

    minus_intensity = minus_intensity_full[
        :, :, flank:-flank
    ]


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("FULL STRAND-AWARE LOCAL + BPE FORWARD")
print("=" * 80)

print("FULL outputs:")
print(" plus presence: ", plus_presence_full.shape)
print(" minus presence:", minus_presence_full.shape)

print("\nCENTRAL TARGET outputs:")
print(" plus presence: ", plus_presence.shape)
print(" plus intensity:", plus_intensity.shape)
print(" minus presence:", minus_presence.shape)
print(" minus intensity:", minus_intensity.shape)

print()

expected = (
    B,
    1,
    profile_config.target_length,
)

print(
    "Correct target shape:",
    plus_presence.shape == expected
    and plus_intensity.shape == expected
    and minus_presence.shape == expected
    and minus_intensity.shape == expected
)

FULL STRAND-AWARE LOCAL + BPE FORWARD
FULL outputs:
 plus presence:  torch.Size([4, 1, 1284])
 minus presence: torch.Size([4, 1, 1284])

CENTRAL TARGET outputs:
 plus presence:  torch.Size([4, 1, 1024])
 plus intensity: torch.Size([4, 1, 1024])
 minus presence: torch.Size([4, 1, 1024])
 minus intensity: torch.Size([4, 1, 1024])

Correct target shape: True


In [18]:
# ============================================================
# 5K. WIDE GLOBAL WINDOW ДЛЯ BPE BRANCH
# LOCAL = 1284 bp
# GLOBAL = 9216 bp
# ============================================================

GLOBAL_FLANK = 4096

LOCAL_FLANK = profile_config.context_flank
TARGET_LENGTH = profile_config.target_length

GLOBAL_LENGTH = (
    GLOBAL_FLANK
    + TARGET_LENGTH
    + GLOBAL_FLANK
)

LOCAL_LENGTH = (
    LOCAL_FLANK
    + TARGET_LENGTH
    + LOCAL_FLANK
)


# ============================================================
# Положение LOCAL window внутри GLOBAL window
# ============================================================

LOCAL_START_IN_GLOBAL = (
    GLOBAL_FLANK
    -
    LOCAL_FLANK
)

LOCAL_END_IN_GLOBAL = (
    LOCAL_START_IN_GLOBAL
    +
    LOCAL_LENGTH
)


# ============================================================
# Безопасное извлечение DNA:
# если окно выходит за contig -> PAD кодом N = 8
# ============================================================

def extract_raw_window_torch(
    contig,
    start,
    end,
):
    contig_seq = sequences[contig]
    contig_length = len(contig_seq)

    wanted_length = end - start

    clipped_start = max(start, 0)
    clipped_end = min(end, contig_length)

    core = torch.as_tensor(
        contig_seq[clipped_start:clipped_end],
        dtype=torch.long,
    ).clone()

    left_pad = max(0, -start)
    right_pad = max(0, end - contig_length)

    if left_pad > 0:
        core = torch.cat(
            [
                torch.full(
                    (left_pad,),
                    8,
                    dtype=torch.long,
                ),
                core,
            ]
        )

    if right_pad > 0:
        core = torch.cat(
            [
                core,
                torch.full(
                    (right_pad,),
                    8,
                    dtype=torch.long,
                ),
            ]
        )

    assert core.numel() == wanted_length

    return core


# ============================================================
# Проверяем GLOBAL window на первом tile
# ============================================================

tile = train_batch.tiles.iloc[0]

contig = str(tile["contig"])

target_start = int(tile["target_start"])
target_end = int(tile["target_end"])


global_start = (
    target_start
    -
    GLOBAL_FLANK
)

global_end = (
    target_end
    +
    GLOBAL_FLANK
)


raw_global = extract_raw_window_torch(
    contig,
    global_start,
    global_end,
)


# ============================================================
# Из GLOBAL достаём центральный LOCAL участок
# ============================================================

raw_local_from_global = raw_global[
    LOCAL_START_IN_GLOBAL:
    LOCAL_END_IN_GLOBAL
]


# Тот самый local window, который мы использовали раньше
raw_local_direct = extract_raw_window_torch(
    contig,
    target_start - LOCAL_FLANK,
    target_end + LOCAL_FLANK,
)


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("LOCAL / GLOBAL WINDOW ALIGNMENT")
print("=" * 80)

print(f"Target length:           {TARGET_LENGTH:,} bp")
print(f"Local flank:             {LOCAL_FLANK:,} bp")
print(f"Local window:            {LOCAL_LENGTH:,} bp")

print()

print(f"Global flank:            {GLOBAL_FLANK:,} bp")
print(f"Global window:           {GLOBAL_LENGTH:,} bp")

print()

print(
    "LOCAL inside GLOBAL:",
    f"[{LOCAL_START_IN_GLOBAL:,}, {LOCAL_END_IN_GLOBAL:,})"
)

print()

print("raw_global shape:        ", raw_global.shape)
print("raw_local_from_global:   ", raw_local_from_global.shape)
print("raw_local_direct:        ", raw_local_direct.shape)

print()

print(
    "LOCAL alignment exact:",
    torch.equal(
        raw_local_from_global,
        raw_local_direct,
    )
)

LOCAL / GLOBAL WINDOW ALIGNMENT
Target length:           1,024 bp
Local flank:             130 bp
Local window:            1,284 bp

Global flank:            4,096 bp
Global window:           9,216 bp

LOCAL inside GLOBAL: [3,966, 5,250)

raw_global shape:         torch.Size([9216])
raw_local_from_global:    torch.Size([1284])
raw_local_direct:         torch.Size([1284])

LOCAL alignment exact: True


In [19]:
# ============================================================
# 5L. TOKENIZE WIDE GLOBAL WINDOW
# и найдём BPE-токены, соответствующие LOCAL region
# ============================================================

global_encoded = encode_raw_bpe(raw_global)

global_tokens = global_encoded["tokens"]
global_lengths = global_encoded["lengths"]
global_softmask = global_encoded["softmask_fraction"]
global_bp_to_token = global_encoded["bp_to_token"]


# ============================================================
# Какие GLOBAL BPE tokens соответствуют нашим LOCAL 1284 bp?
# ============================================================

local_bp_to_global_token = global_bp_to_token[
    LOCAL_START_IN_GLOBAL:
    LOCAL_END_IN_GLOBAL
]

local_global_token_ids = torch.unique(
    local_bp_to_global_token,
    sorted=True,
)


print("=" * 80)
print("GLOBAL BPE TOKENIZATION")
print("=" * 80)

print(f"Global DNA:              {GLOBAL_LENGTH:,} bp")
print(f"Global BPE tokens:       {global_tokens.numel():,}")

print(
    f"Compression:             "
    f"{GLOBAL_LENGTH / global_tokens.numel():.4f}x"
)

print()
print(
    "Global bp mapping:       ",
    global_bp_to_token.shape,
)

print(
    "LOCAL bp mapping:        ",
    local_bp_to_global_token.shape,
)

print()
print(
    "GLOBAL tokens touching LOCAL:",
    local_global_token_ids.numel(),
)

print(
    "First global token for LOCAL:",
    int(local_global_token_ids[0]),
)

print(
    "Last global token for LOCAL: ",
    int(local_global_token_ids[-1]),
)

print()
print(
    "LOCAL still exactly 1284 bp:",
    local_bp_to_global_token.numel() == LOCAL_LENGTH
)

GLOBAL BPE TOKENIZATION
Global DNA:              9,216 bp
Global BPE tokens:       3,670
Compression:             2.5112x

Global bp mapping:        torch.Size([9216])
LOCAL bp mapping:         torch.Size([1284])

GLOBAL tokens touching LOCAL: 511
First global token for LOCAL: 1585
Last global token for LOCAL:  2095

LOCAL still exactly 1284 bp: True


In [20]:
# ============================================================
# 5M. GLOBAL BPE CNN
#
# 9216 bp
#   ↓ BPE
# ~3670 tokens
#   ↓
# 10 dilated residual blocks
#
# dilations:
# 1,2,4,8,16,32,64,128,256,512
# ============================================================

import torch.nn as nn


GLOBAL_DILATIONS = (
    1, 2, 4, 8, 16,
    32, 64, 128, 256, 512,
)


# ============================================================
# Один residual block
# ============================================================

class GlobalResidualBlock(nn.Module):

    def __init__(
        self,
        channels: int,
        dilation: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        x,
        token_mask,
    ):

        residual = x

        x = self.conv1(x)
        x = self.activation(x)

        x = self.conv2(x)

        # residual connection
        x = x + residual
        x = self.activation(x)

        # padded BPE positions должны оставаться нулевыми
        x = (
            x
            *
            token_mask.unsqueeze(1)
        )

        return x


# ============================================================
# Полный GLOBAL encoder
# ============================================================

class GlobalBPEEncoder(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        embedding_dim: int = 16,
        channels: int = 32,
        max_token_length: int = 4,
        dilations=GLOBAL_DILATIONS,
    ):
        super().__init__()

        self.max_token_length = max_token_length
        self.dilations = tuple(dilations)

        # ----------------------------------------------------
        # TOKEN ID -> embedding
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size + 1,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id,
        )

        # embedding 16
        # + softmask 1
        # + token length 1
        #
        # = 18 channels
        input_channels = embedding_dim + 2

        # ----------------------------------------------------
        # 18 -> 32 channels
        # ----------------------------------------------------

        self.input_projection = nn.Sequential(
            nn.Conv1d(
                input_channels,
                channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        # ----------------------------------------------------
        # Dilated residual tower
        # ----------------------------------------------------

        self.blocks = nn.ModuleList(
            [
                GlobalResidualBlock(
                    channels=channels,
                    dilation=d,
                )
                for d in self.dilations
            ]
        )


    def forward(
        self,
        token_ids,
        token_lengths,
        softmask_fraction,
        token_mask,
    ):

        # ====================================================
        # EMBEDDING
        # ====================================================

        embedding = self.embedding(
            token_ids
        )

        # [B, T, 16]


        # ====================================================
        # EXTRA FEATURES
        # ====================================================

        length_feature = (
            token_lengths.float()
            /
            self.max_token_length
        ).unsqueeze(-1)

        softmask_feature = (
            softmask_fraction
            .unsqueeze(-1)
        )


        # [B, T, 18]
        x = torch.cat(
            [
                embedding,
                softmask_feature,
                length_feature,
            ],
            dim=-1,
        )


        # PAD -> 0
        x = (
            x
            *
            token_mask.unsqueeze(-1)
        )


        # Conv1d:
        # [B, channels, T]
        x = x.transpose(1, 2)


        # ====================================================
        # INPUT PROJECTION
        # ====================================================

        x = self.input_projection(x)

        x = (
            x
            *
            token_mask.unsqueeze(1)
        )


        # ====================================================
        # GLOBAL DILATED CNN
        # ====================================================

        for block in self.blocks:

            x = block(
                x,
                token_mask,
            )


        return x


# ============================================================
# Создаём GLOBAL encoder
# ============================================================

global_bpe_encoder = GlobalBPEEncoder(
    vocab_size=vocab_size,
    pad_token_id=PAD_TOKEN_ID,
    embedding_dim=16,
    channels=32,
    max_token_length=max(token_length.values()),
)


# ============================================================
# Готовим НАШ wide window
#
# [3670] -> [1, 3670]
# ============================================================

wide_token_ids = (
    global_tokens
    .unsqueeze(0)
)

wide_token_lengths = (
    global_lengths
    .unsqueeze(0)
)

wide_softmask = (
    global_softmask
    .unsqueeze(0)
)

wide_token_mask = torch.ones_like(
    wide_token_ids,
    dtype=torch.bool,
)


# ============================================================
# FORWARD
# ============================================================

with torch.no_grad():

    wide_global_features = global_bpe_encoder(
        wide_token_ids,
        wide_token_lengths,
        wide_softmask,
        wide_token_mask,
    )


# ============================================================
# RECEPTIVE FIELD
# ============================================================

rf_tokens = (
    1
    +
    4 * sum(GLOBAL_DILATIONS)
)

average_bp_per_token = (
    GLOBAL_LENGTH
    /
    global_tokens.numel()
)

rf_bp_approx = (
    rf_tokens
    *
    average_bp_per_token
)


# ============================================================
# PARAMETERS
# ============================================================

parameter_count = sum(
    p.numel()
    for p in global_bpe_encoder.parameters()
)


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("GLOBAL BPE CNN")
print("=" * 80)

print(
    "Input tokens:             ",
    wide_token_ids.shape,
)

print(
    "Output features:          ",
    wide_global_features.shape,
)

print()

print(
    "Residual blocks:          ",
    len(GLOBAL_DILATIONS),
)

print(
    "Dilations:                ",
    GLOBAL_DILATIONS,
)

print()

print(
    f"Theoretical RF:           "
    f"{rf_tokens:,} BPE tokens"
)

print(
    f"Approx theoretical RF:    "
    f"{rf_bp_approx:,.0f} bp"
)

print(
    f"Actual available window:  "
    f"{GLOBAL_LENGTH:,} bp"
)

print()

print(
    "Covers complete window:   ",
    rf_tokens >= global_tokens.numel(),
)

print(
    f"Parameters:               "
    f"{parameter_count:,}"
)

GLOBAL BPE CNN
Input tokens:              torch.Size([1, 3670])
Output features:           torch.Size([1, 32, 3670])

Residual blocks:           10
Dilations:                 (1, 2, 4, 8, 16, 32, 64, 128, 256, 512)

Theoretical RF:           4,093 BPE tokens
Approx theoretical RF:    10,278 bp
Actual available window:  9,216 bp

Covers complete window:    True
Parameters:               63,584


In [21]:
# ============================================================
# 5N. GLOBAL BPE FEATURES -> CENTRAL LOCAL 1284 bp
# ============================================================

# Для каждой из центральных 1284 bp уже известно,
# какому GLOBAL BPE token она принадлежит:
#
# local_bp_to_global_token.shape == [1284]

gather_index = (
    local_bp_to_global_token
    .unsqueeze(0)      # [1, 1284]
    .unsqueeze(1)      # [1, 1, 1284]
    .expand(
        -1,
        wide_global_features.shape[1],
        -1,
    )
)

# ------------------------------------------------------------
# [1, 32, 3670]
#       ↓ gather
# [1, 32, 1284]
# ------------------------------------------------------------

global_features_for_local = torch.gather(
    wide_global_features,
    dim=2,
    index=gather_index,
)


print("=" * 80)
print("GLOBAL -> LOCAL FEATURE ALIGNMENT")
print("=" * 80)

print(
    "Full global features:    ",
    wide_global_features.shape,
)

print(
    "LOCAL bp->token mapping: ",
    local_bp_to_global_token.shape,
)

print(
    "Features for LOCAL:      ",
    global_features_for_local.shape,
)

print()

print(
    "Correct LOCAL length:",
    global_features_for_local.shape[-1] == LOCAL_LENGTH
)

print(
    "Correct channels:",
    global_features_for_local.shape[1] == 32
)

GLOBAL -> LOCAL FEATURE ALIGNMENT
Full global features:     torch.Size([1, 32, 3670])
LOCAL bp->token mapping:  torch.Size([1284])
Features for LOCAL:       torch.Size([1, 32, 1284])

Correct LOCAL length: True
Correct channels: True


In [22]:
# ============================================================
# 5O. WIDE GLOBAL BPE — FULL PLUS BATCH
# ============================================================

def build_wide_global_bpe_batch(
    tiles,
    reverse_complement=False,
):
    encoded = []

    for i in range(len(tiles)):

        tile = tiles.iloc[i]

        contig = str(tile["contig"])

        target_start = int(tile["target_start"])
        target_end = int(tile["target_end"])

        # ----------------------------------------------------
        # GLOBAL 9216 bp
        # ----------------------------------------------------

        global_start = target_start - GLOBAL_FLANK
        global_end = target_end + GLOBAL_FLANK

        raw = extract_raw_window_torch(
            contig,
            global_start,
            global_end,
        )

        if reverse_complement:
            raw = RC_LOOKUP[raw].flip(0)

        item = encode_raw_bpe(raw)

        # ----------------------------------------------------
        # Для каждой LOCAL bp узнаём индекс GLOBAL token
        # ----------------------------------------------------

        local_bp_to_token = item["bp_to_token"][
            LOCAL_START_IN_GLOBAL:
            LOCAL_END_IN_GLOBAL
        ]

        item["local_bp_to_token"] = local_bp_to_token

        encoded.append(item)


    # ========================================================
    # PAD до максимального числа BPE tokens в batch
    # ========================================================

    batch_size = len(encoded)

    max_tokens = max(
        item["tokens"].numel()
        for item in encoded
    )

    token_ids = torch.full(
        (batch_size, max_tokens),
        PAD_TOKEN_ID,
        dtype=torch.long,
    )

    token_lengths = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.long,
    )

    softmask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.float32,
    )

    token_mask = torch.zeros(
        (batch_size, max_tokens),
        dtype=torch.bool,
    )

    local_bp_to_token = torch.zeros(
        (batch_size, LOCAL_LENGTH),
        dtype=torch.long,
    )


    for i, item in enumerate(encoded):

        n = item["tokens"].numel()

        token_ids[i, :n] = item["tokens"]
        token_lengths[i, :n] = item["lengths"]
        softmask[i, :n] = item["softmask_fraction"]
        token_mask[i, :n] = True

        local_bp_to_token[i] = (
            item["local_bp_to_token"]
        )


    return {
        "token_ids": token_ids,
        "token_lengths": token_lengths,
        "softmask": softmask,
        "token_mask": token_mask,
        "local_bp_to_token": local_bp_to_token,
    }


# ============================================================
# PLUS GLOBAL BATCH
# ============================================================

global_plus_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=False,
)


# ============================================================
# GLOBAL CNN
# ============================================================

with torch.no_grad():

    global_plus_features = global_bpe_encoder(
        global_plus_batch["token_ids"],
        global_plus_batch["token_lengths"],
        global_plus_batch["softmask"],
        global_plus_batch["token_mask"],
    )


# ============================================================
# Берём признаки центральных 1284 bp
# ============================================================

gather_index = (
    global_plus_batch["local_bp_to_token"]
    .unsqueeze(1)
    .expand(
        -1,
        global_plus_features.shape[1],
        -1,
    )
)

global_plus_local_features = torch.gather(
    global_plus_features,
    dim=2,
    index=gather_index,
)


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("WIDE GLOBAL BPE — PLUS BATCH")
print("=" * 80)

print(
    "Global token batch:       ",
    global_plus_batch["token_ids"].shape,
)

print(
    "Valid tokens per window:  ",
    global_plus_batch["token_mask"].sum(dim=1),
)

print()

print(
    "Global CNN output:        ",
    global_plus_features.shape,
)

print(
    "Central LOCAL features:   ",
    global_plus_local_features.shape,
)

print()

print(
    "Expected [B,32,1284]:",
    global_plus_local_features.shape
    ==
    (len(train_batch.tiles), 32, LOCAL_LENGTH)
)

WIDE GLOBAL BPE — PLUS BATCH
Global token batch:        torch.Size([4, 3708])
Valid tokens per window:   tensor([3670, 3637, 3556, 3708])

Global CNN output:         torch.Size([4, 32, 3708])
Central LOCAL features:    torch.Size([4, 32, 1284])

Expected [B,32,1284]: True


In [23]:
# ============================================================
# 5P. FIX MASKING INSIDE GLOBAL RESIDUAL BLOCK
# ============================================================

class GlobalResidualBlock(nn.Module):

    def __init__(
        self,
        channels: int,
        dilation: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        x,
        token_mask,
    ):

        mask = token_mask.unsqueeze(1)

        residual = x

        # ----------------------------------------------------
        # Conv 1
        # ----------------------------------------------------

        x = self.conv1(x)
        x = self.activation(x)

        # PAD не даём существовать даже между convolutions
        x = x * mask

        # ----------------------------------------------------
        # Conv 2
        # ----------------------------------------------------

        x = self.conv2(x)

        x = x + residual
        x = self.activation(x)

        # снова убираем PAD
        x = x * mask

        return x

In [24]:
global_bpe_encoder = GlobalBPEEncoder(
    vocab_size=vocab_size,
    pad_token_id=PAD_TOKEN_ID,
    embedding_dim=16,
    channels=32,
    max_token_length=max(token_length.values()),
)

print(
    "Global encoder recreated:",
    sum(p.numel() for p in global_bpe_encoder.parameters())
)

Global encoder recreated: 63584


In [25]:
# ============================================================
# 5Q. WIDE GLOBAL BPE — REVERSE-COMPLEMENT BATCH
# ============================================================

global_rc_batch = build_wide_global_bpe_batch(
    train_batch.tiles,
    reverse_complement=True,
)


# ============================================================
# GLOBAL CNN — те же самые веса
# ============================================================

with torch.no_grad():

    global_rc_features = global_bpe_encoder(
        global_rc_batch["token_ids"],
        global_rc_batch["token_lengths"],
        global_rc_batch["softmask"],
        global_rc_batch["token_mask"],
    )


# ============================================================
# Выбираем признаки для центральных LOCAL 1284 bp
# ============================================================

rc_gather_index = (
    global_rc_batch["local_bp_to_token"]
    .unsqueeze(1)
    .expand(
        -1,
        global_rc_features.shape[1],
        -1,
    )
)

global_rc_local_features = torch.gather(
    global_rc_features,
    dim=2,
    index=rc_gather_index,
)


# ============================================================
# Дополнительная проверка:
# RC central DNA из wide-window должна совпадать
# с x_both_strands[B:]
# ============================================================

tile0 = train_batch.tiles.iloc[0]

contig0 = str(tile0["contig"])
target_start0 = int(tile0["target_start"])
target_end0 = int(tile0["target_end"])

raw_global0 = extract_raw_window_torch(
    contig0,
    target_start0 - GLOBAL_FLANK,
    target_end0 + GLOBAL_FLANK,
)

raw_global0_rc = RC_LOOKUP[raw_global0].flip(0)

raw_local0_rc = raw_global0_rc[
    LOCAL_START_IN_GLOBAL:
    LOCAL_END_IN_GLOBAL
]

collapsed_local0_rc = BASE_LOOKUP[
    raw_local0_rc
]

cnn_local0_rc = (
    train_batch.x_both_strands[
        len(train_batch.tiles)
    ]
    .argmax(dim=0)
    .long()
    .cpu()
)


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("WIDE GLOBAL BPE — RC BATCH")
print("=" * 80)

print(
    "Global RC token batch:    ",
    global_rc_batch["token_ids"].shape,
)

print(
    "Valid tokens per window:  ",
    global_rc_batch["token_mask"].sum(dim=1),
)

print()

print(
    "Global RC CNN output:     ",
    global_rc_features.shape,
)

print(
    "Central RC features:      ",
    global_rc_local_features.shape,
)

print()

print(
    "Expected [B,32,1284]:",
    global_rc_local_features.shape
    ==
    (len(train_batch.tiles), 32, LOCAL_LENGTH)
)

print(
    "RC central DNA matches local CNN:",
    torch.equal(
        collapsed_local0_rc,
        cnn_local0_rc,
    )
)

WIDE GLOBAL BPE — RC BATCH
Global RC token batch:     torch.Size([4, 3709])
Valid tokens per window:   tensor([3570, 3668, 3611, 3709])

Global RC CNN output:      torch.Size([4, 32, 3709])
Central RC features:       torch.Size([4, 32, 1284])

Expected [B,32,1284]: True
RC central DNA matches local CNN: True


In [26]:
# ============================================================
# 5R. FINAL LOCAL + WIDE GLOBAL BPE MODEL
# ============================================================

class CNNPresenceIntensityWideBPE(nn.Module):

    def __init__(
        self,
        local_model,
        global_encoder,
        local_channels=32,
        global_channels=32,
    ):
        super().__init__()

        # ----------------------------------------------------
        # LOCAL branch
        # ----------------------------------------------------

        self.backbone = local_model.backbone

        # ----------------------------------------------------
        # GLOBAL BPE branch
        # ----------------------------------------------------

        self.global_encoder = global_encoder

        # ----------------------------------------------------
        # Fusion:
        #
        # 32 local + 32 global = 64
        #              ↓
        #             32
        # ----------------------------------------------------

        self.fusion = nn.Sequential(
            nn.Conv1d(
                local_channels + global_channels,
                local_channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        # ----------------------------------------------------
        # Presence + intensity
        # ----------------------------------------------------

        self.profile_head = nn.Conv1d(
            local_channels,
            2,
            kernel_size=1,
        )


    def forward(
        self,
        x_local,
        global_token_ids,
        global_token_lengths,
        global_softmask,
        global_token_mask,
        local_bp_to_global_token,
    ):

        # ====================================================
        # LOCAL BRANCH
        # ====================================================

        local_features = self.backbone(
            x_local
        )

        # [B, 32, 1284]


        # ====================================================
        # GLOBAL BRANCH
        # ====================================================

        global_features = self.global_encoder(
            global_token_ids,
            global_token_lengths,
            global_softmask,
            global_token_mask,
        )

        # [B, 32, ~3700]


        # ====================================================
        # GLOBAL tokens -> центральные 1284 bp
        # ====================================================

        gather_index = (
            local_bp_to_global_token
            .unsqueeze(1)
            .expand(
                -1,
                global_features.shape[1],
                -1,
            )
        )

        global_local_features = torch.gather(
            global_features,
            dim=2,
            index=gather_index,
        )

        # [B, 32, 1284]


        # ====================================================
        # FUSION
        # ====================================================

        combined = torch.cat(
            [
                local_features,
                global_local_features,
            ],
            dim=1,
        )

        # [B, 64, 1284]

        fused = self.fusion(
            combined
        )

        # [B, 32, 1284]


        # ====================================================
        # HEAD
        # ====================================================

        outputs = self.profile_head(
            fused
        )

        presence = outputs[:, 0:1, :]
        intensity = outputs[:, 1:2, :]

        return presence, intensity


# ============================================================
# Создаём единую модель
# ============================================================

wide_local_model = create_cnn_presence_intensity()

wide_fusion_model = CNNPresenceIntensityWideBPE(
    local_model=wide_local_model,
    global_encoder=global_bpe_encoder,
)


# ============================================================
# PLUS smoke test
# ============================================================

with torch.no_grad():

    plus_presence_full, plus_intensity_full = (
        wide_fusion_model(
            train_batch.x,
            global_plus_batch["token_ids"],
            global_plus_batch["token_lengths"],
            global_plus_batch["softmask"],
            global_plus_batch["token_mask"],
            global_plus_batch["local_bp_to_token"],
        )
    )


print("=" * 80)
print("LOCAL + WIDE GLOBAL BPE MODEL")
print("=" * 80)

print(
    "Local input:            ",
    train_batch.x.shape,
)

print(
    "Global BPE input:       ",
    global_plus_batch["token_ids"].shape,
)

print()

print(
    "Presence output:        ",
    plus_presence_full.shape,
)

print(
    "Intensity output:       ",
    plus_intensity_full.shape,
)

print()

print(
    "Correct base resolution:",
    plus_presence_full.shape
    ==
    (len(train_batch.tiles), 1, LOCAL_LENGTH)
)

print()

parameter_count = sum(
    p.numel()
    for p in wide_fusion_model.parameters()
)

print(
    f"Total parameters:       {parameter_count:,}"
)

LOCAL + WIDE GLOBAL BPE MODEL
Local input:             torch.Size([4, 5, 1284])
Global BPE input:        torch.Size([4, 3708])

Presence output:         torch.Size([4, 1, 1284])
Intensity output:        torch.Size([4, 1, 1284])

Correct base resolution: True

Total parameters:       124,802


In [27]:
# ============================================================
# 5S. FULL STRAND-AWARE FORWARD
# LOCAL + WIDE GLOBAL BPE
# ============================================================

B = len(train_batch.tiles)
flank = profile_config.context_flank

x_plus = train_batch.x_both_strands[:B]
x_rc = train_batch.x_both_strands[B:]


with torch.no_grad():

    # ========================================================
    # PLUS
    # ========================================================

    plus_presence_full, plus_intensity_full = wide_fusion_model(
        x_plus,
        global_plus_batch["token_ids"],
        global_plus_batch["token_lengths"],
        global_plus_batch["softmask"],
        global_plus_batch["token_mask"],
        global_plus_batch["local_bp_to_token"],
    )


    # ========================================================
    # REVERSE COMPLEMENT
    # ========================================================

    rc_presence_full, rc_intensity_full = wide_fusion_model(
        x_rc,
        global_rc_batch["token_ids"],
        global_rc_batch["token_lengths"],
        global_rc_batch["softmask"],
        global_rc_batch["token_mask"],
        global_rc_batch["local_bp_to_token"],
    )


    # Возвращаем RC output в координаты исходной DNA
    minus_presence_full = rc_presence_full.flip(-1)
    minus_intensity_full = rc_intensity_full.flip(-1)


    # ========================================================
    # CENTRAL TARGET: 1284 -> 1024
    # ========================================================

    plus_presence = plus_presence_full[:, :, flank:-flank]
    plus_intensity = plus_intensity_full[:, :, flank:-flank]

    minus_presence = minus_presence_full[:, :, flank:-flank]
    minus_intensity = minus_intensity_full[:, :, flank:-flank]


# ============================================================
# CHECK
# ============================================================

expected_shape = (
    B,
    1,
    profile_config.target_length,
)

print("=" * 80)
print("FULL WIDE MODEL — PLUS + RC")
print("=" * 80)

print("PLUS:")
print(" presence: ", plus_presence.shape)
print(" intensity:", plus_intensity.shape)

print()

print("MINUS:")
print(" presence: ", minus_presence.shape)
print(" intensity:", minus_intensity.shape)

print()

print(
    "All target shapes correct:",
    plus_presence.shape == expected_shape
    and plus_intensity.shape == expected_shape
    and minus_presence.shape == expected_shape
    and minus_intensity.shape == expected_shape
)

FULL WIDE MODEL — PLUS + RC
PLUS:
 presence:  torch.Size([4, 1, 1024])
 intensity: torch.Size([4, 1, 1024])

MINUS:
 presence:  torch.Size([4, 1, 1024])
 intensity: torch.Size([4, 1, 1024])

All target shapes correct: True


In [28]:
# ============================================================
# 5T. GRADIENT FLOW SMOKE TEST
# LOCAL + WIDE GLOBAL BPE
# ============================================================

wide_fusion_model.zero_grad(set_to_none=True)


# ============================================================
# FORWARD БЕЗ torch.no_grad()
# ============================================================

test_presence, test_intensity = wide_fusion_model(
    train_batch.x,
    global_plus_batch["token_ids"],
    global_plus_batch["token_lengths"],
    global_plus_batch["softmask"],
    global_plus_batch["token_mask"],
    global_plus_batch["local_bp_to_token"],
)


# Центральные 1024 bp
test_presence = test_presence[
    :, :, LOCAL_FLANK:-LOCAL_FLANK
]

test_intensity = test_intensity[
    :, :, LOCAL_FLANK:-LOCAL_FLANK
]


# ============================================================
# DUMMY LOSS
#
# Только проверяем прохождение градиента.
# Это НЕ training objective.
# ============================================================

dummy_loss = (
    test_presence.square().mean()
    +
    test_intensity.square().mean()
)

dummy_loss.backward()


# ============================================================
# HELPER
# ============================================================

def grad_norm(module):
    total = 0.0
    found = False

    for parameter in module.parameters():

        if parameter.grad is not None:

            found = True

            total += (
                parameter.grad
                .detach()
                .float()
                .square()
                .sum()
                .item()
            )

    if not found:
        return None

    return total ** 0.5


# ============================================================
# CHECK
# ============================================================

print("=" * 80)
print("GRADIENT FLOW")
print("=" * 80)

print(
    f"Dummy loss:              "
    f"{dummy_loss.item():.6f}"
)

print()

print(
    "LOCAL backbone grad:     ",
    grad_norm(
        wide_fusion_model.backbone
    )
)

print(
    "BPE embedding grad:      ",
    grad_norm(
        wide_fusion_model.global_encoder.embedding
    )
)

print(
    "GLOBAL block 1 grad:     ",
    grad_norm(
        wide_fusion_model.global_encoder.blocks[0]
    )
)

print(
    "GLOBAL block 10 grad:    ",
    grad_norm(
        wide_fusion_model.global_encoder.blocks[-1]
    )
)

print(
    "Fusion grad:             ",
    grad_norm(
        wide_fusion_model.fusion
    )
)

print(
    "Profile head grad:       ",
    grad_norm(
        wide_fusion_model.profile_head
    )
)

GRADIENT FLOW
Dummy loss:              0.079389

LOCAL backbone grad:      0.5134389983447795
BPE embedding grad:       0.00048304982426947856
GLOBAL block 1 grad:      0.0013475595522091743
GLOBAL block 10 grad:     0.030363782422566866
Fusion grad:              0.544919404499374
Profile head grad:        0.5909092103267762


## Двухголовая модель

Backbone полностью совпадает с CNN-EXP-001. Добавляется только второй выходной канал.

In [30]:
# ============================================================
# DIRECTML FIX:
# bool mask -> activation dtype BEFORE multiplication
# ============================================================

class GlobalResidualBlock(nn.Module):

    def __init__(
        self,
        channels: int,
        dilation: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            dilation=dilation,
            padding=dilation,
        )

        self.activation = nn.GELU()

    def forward(
        self,
        x,
        token_mask,
    ):

        # ВАЖНО для DirectML:
        # bool -> тот же dtype, что у activation
        mask = (
            token_mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )

        residual = x

        x = self.conv1(x)
        x = self.activation(x)

        x = x * mask

        x = self.conv2(x)

        x = x + residual
        x = self.activation(x)

        x = x * mask

        return x


class GlobalBPEEncoder(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        pad_token_id: int,
        embedding_dim: int = 16,
        channels: int = 32,
        max_token_length: int = 4,
        dilations=GLOBAL_DILATIONS,
    ):
        super().__init__()

        self.max_token_length = max_token_length
        self.dilations = tuple(dilations)

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size + 1,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id,
        )

        input_channels = embedding_dim + 2

        self.input_projection = nn.Sequential(
            nn.Conv1d(
                input_channels,
                channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        self.blocks = nn.ModuleList(
            [
                GlobalResidualBlock(
                    channels=channels,
                    dilation=d,
                )
                for d in self.dilations
            ]
        )

    def forward(
        self,
        token_ids,
        token_lengths,
        softmask_fraction,
        token_mask,
    ):

        # [B,T,E]
        embedding = self.embedding(token_ids)

        length_feature = (
            token_lengths.float()
            / self.max_token_length
        ).unsqueeze(-1)

        softmask_feature = (
            softmask_fraction.float()
            .unsqueeze(-1)
        )

        x = torch.cat(
            [
                embedding,
                softmask_feature,
                length_feature,
            ],
            dim=-1,
        )

        # ----------------------------------------------------
        # DIRECTML FIX №1
        # ----------------------------------------------------

        mask_bt = (
            token_mask
            .unsqueeze(-1)
            .to(dtype=x.dtype)
        )

        x = x * mask_bt

        # [B,T,C] -> [B,C,T]
        x = x.transpose(1, 2)

        x = self.input_projection(x)

        # ----------------------------------------------------
        # DIRECTML FIX №2
        # ----------------------------------------------------

        mask_bct = (
            token_mask
            .unsqueeze(1)
            .to(dtype=x.dtype)
        )

        x = x * mask_bct

        for block in self.blocks:
            x = block(
                x,
                token_mask,
            )

        return x


print("DirectML-safe GlobalBPEEncoder defined")

DirectML-safe GlobalBPEEncoder defined


In [ ]:
# ============================================================
# LOCAL + WIDE GLOBAL BPE MODEL — DIRECTML SMOKE TEST
# ============================================================

B = len(train_batch.tiles)

# ------------------------------------------------------------
# Создаём модель С НУЛЯ
# ------------------------------------------------------------

multitask_model = CNNPresenceIntensityWideBPE(
    local_model=create_cnn_presence_intensity(),
    global_encoder=GlobalBPEEncoder(
        vocab_size=vocab_size,
        pad_token_id=PAD_TOKEN_ID,
        embedding_dim=16,
        channels=32,
        max_token_length=max(token_length.values()),
    ),
).to(device)

multitask_model.eval()


# ============================================================
# LOCAL inputs
# ============================================================

x_plus_gpu = (
    train_batch.x_both_strands[:B]
    .to(device)
)

x_rc_gpu = (
    train_batch.x_both_strands[B:]
    .to(device)
)


# ============================================================
# GLOBAL PLUS inputs
# ============================================================

global_plus_gpu = {
    key: value.to(device)
    for key, value in global_plus_batch.items()
}


# ============================================================
# GLOBAL RC inputs
# ============================================================

global_rc_gpu = {
    key: value.to(device)
    for key, value in global_rc_batch.items()
}


# ============================================================
# FORWARD
# ============================================================

with torch.no_grad():

    plus_presence_full, plus_intensity_full = (
        multitask_model(
            x_plus_gpu,
            global_plus_gpu["token_ids"],
            global_plus_gpu["token_lengths"],
            global_plus_gpu["softmask"],
            global_plus_gpu["token_mask"],
            global_plus_gpu["local_bp_to_token"],
        )
    )

    rc_presence_full, rc_intensity_full = (
        multitask_model(
            x_rc_gpu,
            global_rc_gpu["token_ids"],
            global_rc_gpu["token_lengths"],
            global_rc_gpu["softmask"],
            global_rc_gpu["token_mask"],
            global_rc_gpu["local_bp_to_token"],
        )
    )


# ============================================================
# Возвращаем RC в genomic coordinates
# ============================================================

minus_presence_full = rc_presence_full.flip(-1)
minus_intensity_full = rc_intensity_full.flip(-1)


# ============================================================
# CENTRAL TARGET CROP
# 1284 -> 1024
# ============================================================

flank = profile_config.context_flank

plus_presence = plus_presence_full[
    :, :, flank:-flank
]

plus_intensity = plus_intensity_full[
    :, :, flank:-flank
]

minus_presence = minus_presence_full[
    :, :, flank:-flank
]

minus_intensity = minus_intensity_full[
    :, :, flank:-flank
]


# ============================================================
# CHECK
# ============================================================

parameter_count = sum(
    parameter.numel()
    for parameter in multitask_model.parameters()
    if parameter.requires_grad
)

expected_shape = (
    B,
    1,
    profile_config.target_length,
)


print("=" * 80)
print("WIDE BPE MODEL — DIRECTML")
print("=" * 80)

print("Device:", device)

print()

print("Local PLUS input:")
print(" ", x_plus_gpu.shape, x_plus_gpu.device)

print("Global PLUS input:")
print(
    " ",
    global_plus_gpu["token_ids"].shape,
    global_plus_gpu["token_ids"].device,
)

print()

print("PLUS presence: ", plus_presence.shape)
print("PLUS intensity:", plus_intensity.shape)

print("MINUS presence:", minus_presence.shape)
print("MINUS intensity:", minus_intensity.shape)

print()

print("Параметров:", parameter_count)

assert plus_presence.shape == expected_shape
assert plus_intensity.shape == expected_shape
assert minus_presence.shape == expected_shape
assert minus_intensity.shape == expected_shape

assert parameter_count == 124_802

In [ ]:
multitask_model.eval()

with torch.no_grad():
    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

print("Plus presence:", plus_presence.shape)
print("Minus presence reverse:", minus_presence_reverse.shape)
print("Plus intensity:", plus_intensity.shape)
print("Minus intensity reverse:", minus_intensity_reverse.shape)

assert plus_presence.shape == (4, 1, 1024)
assert minus_presence_reverse.shape == (4, 1, 1024)
assert plus_intensity.shape == (4, 1, 1024)
assert minus_intensity_reverse.shape == (4, 1, 1024)

In [ ]:
plus_count = (
    train_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    train_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target = torch.log1p(
    plus_count
)

minus_intensity_target_reverse = torch.log1p(
    minus_count_reverse
)

plus_intensity_mask = (
    plus_count > 0
)

minus_intensity_mask_reverse = (
    minus_count_reverse > 0
)

print("Plus count:", plus_count.shape)
print("Minus count reverse:", minus_count_reverse.shape)

print(
    "Положительных plus:",
    int(plus_intensity_mask.sum()),
)

print(
    "Положительных minus:",
    int(minus_intensity_mask_reverse.sum()),
)

print(
    "Диапазон intensity target:",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).min()
    ),
    "—",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).max()
    ),
)

assert torch.equal(
    plus_intensity_mask.float(),
    train_batch.plus_target,
)

assert torch.equal(
    minus_intensity_mask_reverse.float(),
    train_batch.minus_target_reverse,
)

In [ ]:
import warnings


INTENSITY_LOSS_WEIGHT = 0.1

multitask_model.train()

optimizer = DirectMLAdam(
    multitask_model.parameters(),
    lr=1e-3,
)

# Presence targets и веса.
plus_target_gpu = (
    train_batch.plus_target.to(device)
)
minus_target_reverse_gpu = (
    train_batch.minus_target_reverse.to(device)
)

plus_presence_weight_gpu = (
    train_batch.plus_loss_weight.to(device)
)
minus_presence_weight_reverse_gpu = (
    train_batch.minus_loss_weight_reverse.to(device)
)

# Intensity targets.
plus_intensity_target_gpu = (
    plus_intensity_target.to(device)
)
minus_intensity_target_reverse_gpu = (
    minus_intensity_target_reverse.to(device)
)

# Положительные веса в BCE нормированы на 0.5.
# Умножение на 2 превращает их в условное распределение
# только среди положительных position-strand.
plus_intensity_weight_gpu = (
    2.0
    * train_batch.plus_loss_weight
    * plus_intensity_mask.float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * train_batch.minus_loss_weight_reverse
    * minus_intensity_mask_reverse.float()
).to(device)

optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    total_loss.backward()
    optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = (
    multitask_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .item()
)

intensity_gradient = (
    multitask_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .item()
)

print("Presence loss:", presence_loss.item())
print("Intensity loss:", intensity_loss.item())
print("Total loss:", total_loss.item())
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

In [ ]:
import math
import time
import warnings


PILOT_STEPS = 200
PILOT_BATCH_SIZE = 4
PILOT_SEED = 42
LOG_EVERY = 20
INTENSITY_LOSS_WEIGHT = 0.1

torch.manual_seed(PILOT_SEED)

pilot_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=PILOT_SEED,
)

pilot_model = (
    create_cnn_presence_intensity()
    .to(device)
)

pilot_optimizer = DirectMLAdam(
    pilot_model.parameters(),
    lr=1e-3,
)

pilot_model.train()

pilot_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, PILOT_STEPS + 1):
        batch = pilot_generator.sample_batch(
            batch_size=PILOT_BATCH_SIZE
        )

        # Вход обеих ориентаций уже объединён на CPU.
        x_gpu = batch.x_both_strands.to(device)

        # Presence targets.
        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Intensity targets готовим и разворачиваем на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        pilot_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            pilot_model,
            x_gpu,
            batch_size=PILOT_BATCH_SIZE,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT * intensity_loss
        )

        if not math.isfinite(total_loss.item()):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        pilot_optimizer.step()

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.95 * ema_presence
                + 0.05 * presence_value
            )
            ema_intensity = (
                0.95 * ema_intensity
                + 0.05 * intensity_value
            )
            ema_total = (
                0.95 * ema_total
                + 0.05 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        pilot_history.append(
            {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
            }
        )

        if step == 1 or step % LOG_EVERY == 0:
            elapsed = (
                time.perf_counter()
                - started_at
            )

            print(
                f"step={step:3d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

print()
print("Время:", round(elapsed, 2), "сек")
print("Финальный EMA presence:", round(ema_presence, 6))
print("Финальный EMA intensity:", round(ema_intensity, 6))
print("Финальный EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)

assert not fallback_warnings

In [ ]:
from pathlib import Path


EXP006_SEED = 42
EXP006_STEPS = 10_0000
EXP006_BATCH_SIZE = 4
EXP006_LOG_EVERY = 250
EXP006_CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}
INTENSITY_LOSS_WEIGHT = 0.1

EXP006_DIR = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP006"
    / "run_001"
)

EXP006_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.manual_seed(EXP006_SEED)

exp002_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=EXP006_SEED,
)

exp002_model = (
    create_cnn_presence_intensity()
    .to(device)
)

exp002_optimizer = DirectMLAdam(
    exp002_model.parameters(),
    lr=1e-3,
)

print("Артефакты:", EXP006_DIR)
print("Параметров:", sum(
    parameter.numel()
    for parameter in exp002_model.parameters()
    if parameter.requires_grad
))
print("Шагов:", EXP006_STEPS)
print("Intensity weight:", INTENSITY_LOSS_WEIGHT)

In [ ]:
import copy
import math
import time
import warnings

import pandas as pd


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def save_exp002_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": "CNN-EXP-006",
        "model_type": "presence_intensity",
        "step": step,
        "model_state_dict": model_state_on_cpu(
            exp002_model
        ),
        "optimizer_state_dict": tree_to_cpu(
            exp002_optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            exp002_generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },
        "training_config": {
            "seed": EXP006_SEED,
            "steps": EXP006_STEPS,
            "batch_size": EXP006_BATCH_SIZE,
            "learning_rate": 1e-3,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },
        "seen_position_strand": (
            seen_position_strand
        ),
        "elapsed_seconds": elapsed_seconds,
        "history": list(history),
    }

    torch.save(checkpoint, path)


exp002_model.train()

exp002_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, EXP006_STEPS + 1):
        batch = exp002_generator.sample_batch(
            batch_size=EXP006_BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Всё выравнивание minus выполняется на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        exp002_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            exp002_model,
            x_gpu,
            batch_size=EXP006_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        exp002_optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % EXP006_LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
                "elapsed_seconds": elapsed,
                "seen_position_strand": (
                    seen_position_strand
                ),
                "positions_per_second": (
                    seen_position_strand / elapsed
                ),
            }

            exp002_history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in EXP006_CHECKPOINT_STEPS:
            checkpoint_path = (
                EXP006_DIR
                / f"model_step_{step:05d}.pt"
            )

            save_exp002_checkpoint(
                checkpoint_path,
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=exp002_history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

final_checkpoint_path = (
    EXP006_DIR / "exp002_final.pt"
)
history_path = (
    EXP006_DIR / "training_history.csv"
)

save_exp002_checkpoint(
    final_checkpoint_path,
    step=EXP006_STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=exp002_history,
)

pd.DataFrame(exp002_history).to_csv(
    history_path,
    index=False,
)

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время шага:",
    round(elapsed / EXP006_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", final_checkpoint_path)
print("История:", history_path)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

In [ ]:
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
    load_cnn_presence_intensity_checkpoint,
)


EXP006_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP006"
    / "run_001"
    / "exp002_final.pt"
)

loaded_exp002 = (
    load_cnn_presence_intensity_checkpoint(
        EXP006_CHECKPOINT_PATH,
        device=device,
    )
)

validation_model = loaded_exp002.model
validation_config = loaded_exp002.profile_config
checkpoint = loaded_exp002.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)
print(
    "Device:",
    next(validation_model.parameters()).device,
)

In [ ]:
validation_result = (
    evaluate_presence_intensity_model(
        validation_model,
        validation_generator,
        device=device,
        batch_size=8,
        score_decimals=5,
        progress_every_batches=500,
    )
)

In [ ]:
VALIDATION_METRICS_PATH = (
    EXP006_CHECKPOINT_PATH.parent
    / "validation_metrics.csv"
)

validation_result.metrics.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

overall_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        == "overall"
    ]
    .reset_index(drop=True)
)

display(overall_metrics)

print(
    "Validation time:",
    round(validation_result.elapsed_seconds, 2),
    "sec",
)
print(
    "Position-strand/s:",
    f"{validation_result.positions_per_second:,.0f}",
)
print(
    "CPU fallback:",
    list(validation_result.fallback_warnings),
)
print("Metrics:", VALIDATION_METRICS_PATH)

In [ ]:
per_contig_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        != "overall"
    ]
    .sort_values(
        ["segment", "score"]
    )
    .reset_index(drop=True)
)

display(per_contig_metrics)

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "RF++ охват очень сильно влияет нужно смотреть шире!"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='RF++',
        hypothesis='увеличиваю ещё сильнее RF до максимума',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_rf.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-003',
        reference_metrics='artifacts/experiments/CNN_EXP003/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "увеличенное число шагов сильно улучшает AP"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='STEP++ exp',
        hypothesis='увеличение шагов даст  явный прирост AP',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_step_exp.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-004',
        reference_metrics='artifacts/experiments/CNN_EXP004/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

In [ ]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = False
FINAL_DECISION = "review"  # review / adopt / reject / iterate
FINAL_NOTES = ""

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-006',
        title='BPE test',
        hypothesis='Хочется посмотреть сколько даст сжатая последовательность',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP006_bpe_test.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP006/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-005',
        reference_metrics='artifacts/experiments/CNN_EXP005/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")